In [1]:
# @title MPC
# ============================================================
# Multi-ROI NPZ analysis
# Requested outputs only:
# 1. ROI overlay on full pointcloud
# 2. Max seen temperature across experiment
# 3. Active ROI percent in band
# 4. Active ROI mean temperature with spatial std
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/Nirmal_run_tests/MPC_Contour_Saddle_Manual_tests"

NPZ_PATHS = [
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_1.npz"),
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_2.npz"),
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_3.npz"),
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_4.npz"),
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_05.npz"),
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_6.npz"),
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_7.npz"),
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_8.npz"),
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_9.npz"),
    os.path.join(folder, "05_27_26_saddle_Contour_test_MPC_10.npz"),
]

print("Found", len(NPZ_PATHS), "npz files:")

for p in NPZ_PATHS:
    print(p)
# ============================================================
# User settings
# ============================================================

TARGET_LOW_C = 40.0
TARGET_HIGH_C = 50.0
TARGET_PERCENT_FOR_TIME = 90.0

TEMP_MODE = "auto"

TRIM_START_SEC = 7.0
TRIM_END_SEC = 0.0

AUTO_END_TRIM_ENABLED = False
AUTO_END_TRIM_TARGET_PCT = 90.0

SELECTED_TO_FULL_MAX_DIST_M = 0.015
FRAME_TO_SELECTED_MAX_DIST_M = 0.025
MIN_VALID_FRAC_WARNING = 0.70

FULL_POINT_SIZE = 8
ROI_POINT_SIZE = 45
CMAP = "inferno"

PLOT_SWITCH_LINES = True
PLOT_TARGET_LINE = True

# ============================================================
# Data loading
# ============================================================

def load_multi_roi_npz(npz_path: str):
    data = np.load(npz_path, allow_pickle=True)

    required = ["thermal_frames", "timestamps"]
    for key in required:
        if key not in data.files:
            raise KeyError(f"Missing required key: {key}")

    has_multi_roi = (
        "selected_surface_points_all" in data.files
        and "selected_surface_timestamps_all" in data.files
    )

    if has_multi_roi:
        roi_points_all = list(data["selected_surface_points_all"])
        roi_times_abs = np.asarray(data["selected_surface_timestamps_all"], dtype=np.float64)

        if "selected_surface_roi_index_all" in data.files:
            roi_index_all = np.asarray(data["selected_surface_roi_index_all"], dtype=np.int32)
        else:
            roi_index_all = np.arange(len(roi_points_all), dtype=np.int32)

    elif "selected_surface_points" in data.files:
        roi_points_all = [np.asarray(data["selected_surface_points"], dtype=np.float32)]

        if "selected_surface_time_ns" in data.files:
            roi_times_abs = np.asarray(
                [float(data["selected_surface_time_ns"]) * 1e-9],
                dtype=np.float64,
            )
        else:
            roi_times_abs = np.asarray([float(data["timestamps"][0])], dtype=np.float64)

        roi_index_all = np.asarray([0], dtype=np.int32)

    else:
        raise KeyError(
            "Missing selected ROI fields. Expected selected_surface_points_all or selected_surface_points."
        )

    return {
        "npz_path": npz_path,
        "thermal_frames": data["thermal_frames"],
        "timestamps": np.asarray(data["timestamps"], dtype=np.float64),
        "target_frame": str(data["target_frame"]) if "target_frame" in data.files else "unknown",
        "roi_points_all": [np.asarray(x, dtype=np.float32) for x in roi_points_all],
        "roi_times_abs": roi_times_abs,
        "roi_index_all": roi_index_all,
        "raw": data,
    }

# ============================================================
# Temperature helpers
# ============================================================

def convert_raw_temp_to_celsius(raw_temp_values: np.ndarray, temp_mode: str):
    vals = np.asarray(raw_temp_values, dtype=np.float32)

    if temp_mode == "celsius":
        return vals

    if temp_mode == "divide_1000":
        return vals / 1000.0

    if temp_mode == "kelvin_x100":
        return (vals - 27315.0) / 100.0

    if temp_mode == "kelvin":
        return vals - 273.15

    if temp_mode == "auto":
        out = vals.copy()

        mask_kx100 = out > 1.0e4
        mask_k = (~mask_kx100) & (out > 200.0)

        out[mask_kx100] = (out[mask_kx100] - 27315.0) / 100.0
        out[mask_k] = out[mask_k] - 273.15

        return out

    raise ValueError(f"Unsupported TEMP_MODE: {temp_mode}")


def infer_temperature_mode_from_frame(raw_temp_values: np.ndarray):
    vals = np.asarray(raw_temp_values, dtype=np.float32)
    vals = vals[np.isfinite(vals)]

    if vals.size == 0:
        return "celsius"

    med = float(np.median(vals))
    vmax = float(np.max(vals))

    if med > 10000.0 or vmax > 5000.0:
        return "kelvin_x100"

    if med > 200.0:
        return "kelvin"

    return "celsius"


def detect_temperature_mode(dataset, user_temp_mode):
    if user_temp_mode != "auto":
        return user_temp_mode

    for frame in dataset["thermal_frames"]:
        arr = np.asarray(frame, dtype=np.float32)

        if arr.ndim == 2 and arr.shape[1] >= 4 and arr.shape[0] > 0:
            return infer_temperature_mode_from_frame(arr[:, 3])

    return "celsius"


def extract_xyz_temp(frame_entry, temp_mode="celsius"):
    arr = np.asarray(frame_entry, dtype=np.float32)

    if arr.ndim != 2 or arr.shape[1] < 4:
        raise ValueError(f"Expected frame shape (N,4+), got {arr.shape}")

    xyz = arr[:, :3].astype(np.float32)
    temp_c = convert_raw_temp_to_celsius(arr[:, 3], temp_mode)

    return xyz, temp_c.astype(np.float32)

# ============================================================
# Time and ROI helpers
# ============================================================

def make_relative_time_values(timestamps_abs):
    timestamps_abs = np.asarray(timestamps_abs, dtype=np.float64)
    return timestamps_abs - timestamps_abs[0]


def active_roi_index_for_frames(frame_times_abs, roi_times_abs):
    frame_times_abs = np.asarray(frame_times_abs, dtype=np.float64)
    roi_times_abs = np.asarray(roi_times_abs, dtype=np.float64)

    if len(roi_times_abs) == 0:
        return np.zeros(len(frame_times_abs), dtype=np.int32)

    idx = np.searchsorted(roi_times_abs, frame_times_abs, side="right") - 1
    idx = np.clip(idx, 0, len(roi_times_abs) - 1)

    return idx.astype(np.int32)


def get_switch_frame_indices(active_roi_per_frame):
    active_roi_per_frame = np.asarray(active_roi_per_frame, dtype=np.int32)

    if active_roi_per_frame.size <= 1:
        return np.zeros((0,), dtype=np.int32)

    return np.where(active_roi_per_frame[1:] != active_roi_per_frame[:-1])[0] + 1


def get_roi_segments(active_roi_per_frame):
    active_roi_per_frame = np.asarray(active_roi_per_frame, dtype=np.int32)

    if active_roi_per_frame.size == 0:
        return []

    switch_idx = get_switch_frame_indices(active_roi_per_frame)

    starts = np.concatenate([[0], switch_idx])
    ends = np.concatenate([switch_idx, [len(active_roi_per_frame)]])

    segments = []

    for start, end in zip(starts, ends):
        segments.append({
            "start": int(start),
            "end": int(end),
            "roi": int(active_roi_per_frame[start]),
        })

    return segments


def trim_dataset_indices_by_time(timestamps_abs, trim_start_sec=0.0, trim_end_sec=0.0):
    timestamps_abs = np.asarray(timestamps_abs, dtype=np.float64)

    start_time = timestamps_abs[0] + float(trim_start_sec)
    end_time = timestamps_abs[-1] - float(trim_end_sec)

    if end_time <= start_time:
        raise ValueError("Trim window invalid: end_time <= start_time")

    idx = np.where((timestamps_abs >= start_time) & (timestamps_abs <= end_time))[0]

    if idx.size == 0:
        raise ValueError("No data left after time trimming")

    return idx


def apply_frame_index_subset(dataset, idx):
    out = dataset.copy()
    out["thermal_frames"] = dataset["thermal_frames"][idx]
    out["timestamps"] = dataset["timestamps"][idx]
    return out


def find_final_roi_target_index(timestamps_rel, active_roi_per_frame, active_pct_in_band, target_percent):
    final_roi = int(np.max(active_roi_per_frame))
    mask = active_roi_per_frame == final_roi

    reached = np.where(mask & (active_pct_in_band >= float(target_percent)))[0]

    if reached.size == 0:
        return None, None

    idx = int(reached[0])
    return float(timestamps_rel[idx]), idx


def apply_auto_end_trim_for_final_roi(dataset, active_roi_per_frame, active_pct_in_band):
    timestamps_abs = np.asarray(dataset["timestamps"], dtype=np.float64)
    timestamps_rel = make_relative_time_values(timestamps_abs)

    if AUTO_END_TRIM_ENABLED:
        _, ref_idx = find_final_roi_target_index(
            timestamps_rel,
            active_roi_per_frame,
            active_pct_in_band,
            AUTO_END_TRIM_TARGET_PCT,
        )

        if ref_idx is None:
            raise ValueError(
                f"Auto end trim enabled, but final ROI never reached "
                f"{AUTO_END_TRIM_TARGET_PCT:.1f}%."
            )

        reference_time_abs = float(timestamps_abs[ref_idx])
        final_time_abs = reference_time_abs - float(TRIM_END_SEC)

        if final_time_abs < timestamps_abs[0]:
            raise ValueError("End trim removes the entire recording before the first frame.")

        if final_time_abs > timestamps_abs[-1]:
            raise ValueError(
                f"Requested extension goes past recording end. "
                f"Requested final time={final_time_abs:.3f}s, "
                f"raw final time={timestamps_abs[-1]:.3f}s"
            )

        idx = np.where(timestamps_abs <= final_time_abs)[0]

        print(
            f"Auto trim reference: final ROI first reached "
            f"{AUTO_END_TRIM_TARGET_PCT:.1f}% at t={timestamps_rel[ref_idx]:.3f}s"
        )

        return apply_frame_index_subset(dataset, idx), idx

    ref_idx = len(timestamps_abs) - 1
    final_time_abs = float(timestamps_abs[ref_idx]) - float(TRIM_END_SEC)

    idx = np.where(timestamps_abs <= final_time_abs)[0]

    return apply_frame_index_subset(dataset, idx), idx

# ============================================================
# Mapping helpers
# ============================================================

def get_roi_xyz(roi_points):
    arr = np.asarray(roi_points, dtype=np.float32)

    if arr.ndim != 2 or arr.shape[1] < 3:
        raise ValueError(f"Invalid ROI points shape: {arr.shape}")

    return arr[:, :3].astype(np.float32)


def build_full_part_reference(dataset, reference_frame_index=0, temp_mode="celsius"):
    full_xyz, _ = extract_xyz_temp(
        dataset["thermal_frames"][reference_frame_index],
        temp_mode=temp_mode,
    )

    return full_xyz


def map_roi_to_full_reference(full_xyz, roi_xyz, max_dist_m):
    tree = cKDTree(full_xyz)

    dists, idx_full = tree.query(roi_xyz, k=1)
    keep = dists <= float(max_dist_m)

    mask_on_full = np.zeros(full_xyz.shape[0], dtype=bool)
    mask_on_full[np.unique(idx_full[keep])] = True

    return mask_on_full, keep, dists


def map_frames_to_roi_reference(roi_xyz, thermal_frames, max_match_dist_m, temp_mode):
    tree = cKDTree(roi_xyz)

    frame_count = len(thermal_frames)
    node_count = roi_xyz.shape[0]

    temps_hist = np.full((frame_count, node_count), np.nan, dtype=np.float32)
    valid_frac = np.zeros(frame_count, dtype=np.float32)

    for i in range(frame_count):
        xyz_t, temp_t = extract_xyz_temp(thermal_frames[i], temp_mode=temp_mode)

        dists, idx_roi = tree.query(xyz_t, k=1)
        keep = dists <= float(max_match_dist_m)

        if not np.any(keep):
            continue

        idx_keep = idx_roi[keep]
        temp_keep = temp_t[keep]

        temp_sum = np.zeros(node_count, dtype=np.float64)
        temp_count = np.zeros(node_count, dtype=np.int32)

        np.add.at(temp_sum, idx_keep, temp_keep)
        np.add.at(temp_count, idx_keep, 1)

        assigned = temp_count > 0

        temps_hist[i, assigned] = (
            temp_sum[assigned] / temp_count[assigned]
        ).astype(np.float32)

        valid_frac[i] = float(np.mean(assigned))

    return temps_hist, valid_frac


def forward_fill_nan_rows(arr):
    out = arr.copy()
    frame_count, node_count = out.shape

    for j in range(node_count):
        last = np.nan

        for i in range(frame_count):
            if np.isfinite(out[i, j]):
                last = out[i, j]
            elif np.isfinite(last):
                out[i, j] = last

    return out


def back_fill_initial_nans(arr):
    out = arr.copy()
    _, node_count = out.shape

    for j in range(node_count):
        finite_idx = np.where(np.isfinite(out[:, j]))[0]

        if finite_idx.size > 0:
            first = finite_idx[0]
            out[:first, j] = out[first, j]

    return out


def fill_temperature_history(raw_hist):
    out = forward_fill_nan_rows(raw_hist)
    out = back_fill_initial_nans(out)

    if np.isnan(out).any():
        fill_value = np.nanmedian(out)
        out = np.where(np.isnan(out), fill_value, out)

    return out.astype(np.float32)

# ============================================================
# Metrics
# ============================================================

def compute_roi_metrics(temps_hist_filled, target_low_c, target_high_c):
    in_band = (
        (temps_hist_filled >= target_low_c)
        & (temps_hist_filled <= target_high_c)
    )

    return {
        "pct_in_band": 100.0 * np.mean(in_band, axis=1),
        "min_temp": np.nanmin(temps_hist_filled, axis=1),
        "max_temp": np.nanmax(temps_hist_filled, axis=1),
        "mean_temp": np.nanmean(temps_hist_filled, axis=1),
        "std_temp": np.nanstd(temps_hist_filled, axis=1),
        "final_frame_temp": temps_hist_filled[-1],
        "max_seen_temp": np.nanmax(temps_hist_filled, axis=0),
    }


def build_active_metrics(roi_metrics_all, active_roi_per_frame):
    frame_count = len(active_roi_per_frame)

    active_pct = np.zeros(frame_count, dtype=np.float32)
    active_mean = np.zeros(frame_count, dtype=np.float32)
    active_std = np.zeros(frame_count, dtype=np.float32)
    active_min = np.zeros(frame_count, dtype=np.float32)
    active_max = np.zeros(frame_count, dtype=np.float32)

    for i in range(frame_count):
        roi_idx = int(active_roi_per_frame[i])

        active_pct[i] = roi_metrics_all[roi_idx]["pct_in_band"][i]
        active_mean[i] = roi_metrics_all[roi_idx]["mean_temp"][i]
        active_std[i] = roi_metrics_all[roi_idx]["std_temp"][i]
        active_min[i] = roi_metrics_all[roi_idx]["min_temp"][i]
        active_max[i] = roi_metrics_all[roi_idx]["max_temp"][i]

    return {
        "pct_in_band": active_pct,
        "mean_temp": active_mean,
        "std_temp": active_std,
        "min_temp": active_min,
        "max_temp": active_max,
    }


def build_max_seen_temperature_cloud(roi_xyz_all, roi_metrics_all):
    points = []
    values = []
    roi_ids = []

    for roi_idx, roi_xyz in enumerate(roi_xyz_all):
        points.append(roi_xyz)
        values.append(roi_metrics_all[roi_idx]["max_seen_temp"])
        roi_ids.append(np.full(roi_xyz.shape[0], roi_idx, dtype=np.int32))

    return (
        np.vstack(points).astype(np.float32),
        np.concatenate(values).astype(np.float32),
        np.concatenate(roi_ids).astype(np.int32),
    )


def process_time_summary(timestamps_rel, active_roi_per_frame, active_pct):
    final_target_time, final_target_idx = find_final_roi_target_index(
        timestamps_rel,
        active_roi_per_frame,
        active_pct,
        TARGET_PERCENT_FOR_TIME,
    )

    return final_target_time, final_target_idx

# ============================================================
# Plot helpers
# ============================================================

def make_equal_axes_3d(ax, pts):
    x = pts[:, 0]
    y = pts[:, 1]
    z = pts[:, 2]

    x_mid = 0.5 * (x.min() + x.max())
    y_mid = 0.5 * (y.min() + y.max())
    z_mid = 0.5 * (z.min() + z.max())

    span = max(x.max() - x.min(), y.max() - y.min(), z.max() - z.min())
    half = 0.5 * span

    ax.set_xlim(x_mid - half, x_mid + half)
    ax.set_ylim(y_mid - half, y_mid + half)
    ax.set_zlim(z_mid - half, z_mid + half)


def roi_color(roi_idx):
    cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    return cycle[int(roi_idx) % len(cycle)]


def plot_all_roi_overlay(full_xyz, roi_xyz_all):
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(
        full_xyz[:, 0],
        full_xyz[:, 1],
        full_xyz[:, 2],
        s=FULL_POINT_SIZE,
        alpha=0.10,
        label="Full pointcloud",
    )

    for roi_idx, roi_xyz in enumerate(roi_xyz_all):
        ax.scatter(
            roi_xyz[:, 0],
            roi_xyz[:, 1],
            roi_xyz[:, 2],
            s=ROI_POINT_SIZE,
            color=roi_color(roi_idx),
            label=f"ROI {roi_idx}",
        )

    ax.set_title("ROI overlay on full pointcloud")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Z (m)")

    make_equal_axes_3d(ax, full_xyz)

    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_spatial_scalar(points_xyz, values, title, cmap=CMAP, point_size=ROI_POINT_SIZE):
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    sc = ax.scatter(
        points_xyz[:, 0],
        points_xyz[:, 1],
        points_xyz[:, 2],
        c=values,
        cmap=cmap,
        s=point_size,
    )

    ax.set_title(title)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Z (m)")

    make_equal_axes_3d(ax, points_xyz)

    cbar = plt.colorbar(sc, ax=ax, shrink=0.75, pad=0.08)
    cbar.set_label("Temperature (C)")

    plt.tight_layout()
    plt.show()


def plot_active_percent_in_band(
    timestamps_rel,
    active_pct,
    active_roi_per_frame,
    final_target_time=None,
):
    timestamps_rel = np.asarray(timestamps_rel, dtype=np.float64)
    active_pct = np.asarray(active_pct, dtype=np.float32)

    segments = get_roi_segments(active_roi_per_frame)

    plt.figure(figsize=(10, 5.5))

    used_labels = set()

    for seg in segments:
        start = int(seg["start"])
        end = int(seg["end"])
        roi = int(seg["roi"])

        label = f"ROI {roi}" if roi not in used_labels else None
        used_labels.add(roi)

        plt.plot(
            timestamps_rel[start:end],
            active_pct[start:end],
            color=roi_color(roi),
            linewidth=2.5,
            label=label,
        )

    if PLOT_SWITCH_LINES:
        switch_idx = get_switch_frame_indices(active_roi_per_frame)

        for idx in switch_idx:
            prev_roi = int(active_roi_per_frame[idx - 1])
            next_roi = int(active_roi_per_frame[idx])

            plt.axvline(
                timestamps_rel[idx],
                color="black",
                linestyle="--",
                linewidth=1.4,
                alpha=0.8,
            )

            plt.text(
                timestamps_rel[idx],
                102.0,
                f"ROI {prev_roi} to {next_roi}",
                rotation=90,
                va="bottom",
                ha="right",
                fontsize=9,
            )

    if PLOT_TARGET_LINE:
        plt.axhline(
            TARGET_PERCENT_FOR_TIME,
            linestyle=":",
            linewidth=1.6,
            color="black",
            label=f"{TARGET_PERCENT_FOR_TIME:.0f}% target",
        )

    if final_target_time is not None:
        plt.axvline(
            final_target_time,
            linestyle="-.",
            linewidth=2.0,
            color="black",
            label=f"Final ROI target time = {final_target_time:.2f} s",
        )

    plt.xlabel("Time (s)")
    plt.ylabel("Active ROI nodes in band (%)")
    plt.title(f"Active ROI percent in band [{TARGET_LOW_C:.1f}, {TARGET_HIGH_C:.1f}] C")
    plt.ylim(-2, 108)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_active_mean_temp_with_std(timestamps_rel, active_metrics):
    mean_temp = active_metrics["mean_temp"]
    std_temp = active_metrics["std_temp"]

    plt.figure(figsize=(10, 5.5))

    plt.plot(
        timestamps_rel,
        mean_temp,
        linewidth=2.0,
        label="Active ROI mean temperature",
    )

    plt.fill_between(
        timestamps_rel,
        mean_temp - std_temp,
        mean_temp + std_temp,
        alpha=0.25,
        label="Spatial std",
    )

    plt.axhline(
        TARGET_LOW_C,
        linestyle="--",
        linewidth=1.4,
        label="Target low",
    )

    plt.axhline(
        TARGET_HIGH_C,
        linestyle="--",
        linewidth=1.4,
        label="Target high",
    )

    plt.xlabel("Time (s)")
    plt.ylabel("Temperature (C)")
    plt.title("Active ROI mean temperature with spatial standard deviation")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ============================================================
# Analysis workflow
# ============================================================

def prepare_dataset(npz_path):
    dataset = load_multi_roi_npz(npz_path)

    start_idx = trim_dataset_indices_by_time(
        dataset["timestamps"],
        trim_start_sec=TRIM_START_SEC,
        trim_end_sec=0.0,
    )

    dataset = apply_frame_index_subset(dataset, start_idx)

    temp_mode = detect_temperature_mode(dataset, TEMP_MODE)
    print("Temperature mode:", temp_mode)

    roi_xyz_all = [get_roi_xyz(roi) for roi in dataset["roi_points_all"]]
    roi_count = len(roi_xyz_all)

    active_roi_initial = active_roi_index_for_frames(
        dataset["timestamps"],
        dataset["roi_times_abs"],
    )

    full_reference_xyz = build_full_part_reference(
        dataset,
        reference_frame_index=0,
        temp_mode=temp_mode,
    )

    roi_temps_raw_all = []
    roi_temps_filled_all = []
    roi_valid_frac_all = []
    roi_metrics_all = []
    roi_masks_on_full = []

    for roi_idx, roi_xyz in enumerate(roi_xyz_all):
        mask_on_full, keep, dists = map_roi_to_full_reference(
            full_reference_xyz,
            roi_xyz,
            SELECTED_TO_FULL_MAX_DIST_M,
        )

        roi_masks_on_full.append(mask_on_full)

        raw_hist, valid_frac = map_frames_to_roi_reference(
            roi_xyz,
            dataset["thermal_frames"],
            FRAME_TO_SELECTED_MAX_DIST_M,
            temp_mode,
        )

        filled_hist = fill_temperature_history(raw_hist)

        metrics = compute_roi_metrics(
            filled_hist,
            TARGET_LOW_C,
            TARGET_HIGH_C,
        )

        roi_temps_raw_all.append(raw_hist)
        roi_temps_filled_all.append(filled_hist)
        roi_valid_frac_all.append(valid_frac)
        roi_metrics_all.append(metrics)

        print(
            f"ROI {roi_idx}: nodes={roi_xyz.shape[0]}, "
            f"mean valid mapped fraction={float(np.mean(valid_frac)):.3f}, "
            f"min valid fraction={float(np.min(valid_frac)):.3f}"
        )

        if float(np.min(valid_frac)) < MIN_VALID_FRAC_WARNING:
            print(f"Warning: ROI {roi_idx} has poor mapping in some frames.")

    active_metrics_initial = build_active_metrics(
        roi_metrics_all,
        active_roi_initial,
    )

    dataset, end_idx = apply_auto_end_trim_for_final_roi(
        dataset,
        active_roi_initial,
        active_metrics_initial["pct_in_band"],
    )

    active_roi = active_roi_initial[end_idx]

    roi_temps_raw_all = [x[end_idx] for x in roi_temps_raw_all]
    roi_temps_filled_all = [x[end_idx] for x in roi_temps_filled_all]
    roi_valid_frac_all = [x[end_idx] for x in roi_valid_frac_all]

    roi_metrics_all = [
        compute_roi_metrics(x, TARGET_LOW_C, TARGET_HIGH_C)
        for x in roi_temps_filled_all
    ]

    active_metrics = build_active_metrics(
        roi_metrics_all,
        active_roi,
    )

    timestamps_rel = make_relative_time_values(dataset["timestamps"])

    max_points, max_values, max_roi_ids = build_max_seen_temperature_cloud(
        roi_xyz_all,
        roi_metrics_all,
    )

    final_target_time, final_target_idx = process_time_summary(
        timestamps_rel,
        active_roi,
        active_metrics["pct_in_band"],
    )

    return {
        "dataset": dataset,
        "temp_mode": temp_mode,
        "timestamps_rel": timestamps_rel,
        "full_reference_xyz": full_reference_xyz,
        "roi_xyz_all": roi_xyz_all,
        "roi_count": roi_count,
        "roi_masks_on_full": roi_masks_on_full,
        "roi_temps_raw_all": roi_temps_raw_all,
        "roi_temps_filled_all": roi_temps_filled_all,
        "roi_valid_frac_all": roi_valid_frac_all,
        "roi_metrics_all": roi_metrics_all,
        "active_roi": active_roi,
        "active_metrics": active_metrics,
        "max_points": max_points,
        "max_values": max_values,
        "max_roi_ids": max_roi_ids,
        "final_target_time": final_target_time,
        "final_target_idx": final_target_idx,
    }


def print_diagnostics(results):
    dataset = results["dataset"]
    timestamps_rel = results["timestamps_rel"]
    active_metrics = results["active_metrics"]

    print("")
    print("============================================================")
    print("Diagnostics")
    print("============================================================")
    print("Loaded frame count:", len(dataset["thermal_frames"]))
    print("Time duration after trim (s):", float(timestamps_rel[-1]))
    print("Target frame:", dataset["target_frame"])
    print("ROI count:", results["roi_count"])
    print("Max active percent in band:", float(np.max(active_metrics["pct_in_band"])))

    if results["final_target_time"] is not None:
        print(
            f"Final ROI first reached {TARGET_PERCENT_FOR_TIME:.0f}% in band "
            f"at t={results['final_target_time']:.3f}s"
        )
    else:
        print(f"Final ROI did not reach {TARGET_PERCENT_FOR_TIME:.0f}% in band.")


def plot_all_trials_active_percent_in_band(all_results):
    plt.figure(figsize=(12, 6))

    for idx, item in enumerate(all_results):
        results = item["results"]
        label = f"Trial {idx + 1}"

        timestamps_rel = results["timestamps_rel"]
        active_pct = results["active_metrics"]["pct_in_band"]

        reached = np.where(active_pct >= TARGET_PERCENT_FOR_TIME)[0]

        if reached.size > 0:
            t90 = float(timestamps_rel[reached[0]])
            plot_label = f"{label}, t90 = {t90:.2f} s"
        else:
            plot_label = f"{label}, t90 not reached"

        plt.plot(
            timestamps_rel,
            active_pct,
            linewidth=2.0,
            label=plot_label,
        )

    if PLOT_TARGET_LINE:
        plt.axhline(
            TARGET_PERCENT_FOR_TIME,
            linestyle=":",
            linewidth=1.6,
            color="black",
            label=f"{TARGET_PERCENT_FOR_TIME:.0f}% target",
        )

    plt.xlabel("Time (s)")
    plt.ylabel("Active ROI nodes in band (%)")
    plt.title(f"Active ROI percent in band [{TARGET_LOW_C:.1f}, {TARGET_HIGH_C:.1f}] C")
    plt.ylim(-2, 108)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_all_trials_active_mean_temp_with_std(all_results):
    plt.figure(figsize=(12, 6))

    for idx, item in enumerate(all_results):
        results = item["results"]
        label = f"Trial {idx + 1}"

        timestamps_rel = results["timestamps_rel"]
        mean_temp = results["active_metrics"]["mean_temp"]
        std_temp = results["active_metrics"]["std_temp"]

        line, = plt.plot(
            timestamps_rel,
            mean_temp,
            linewidth=2.0,
            label=f"{label} mean",
        )

        color = line.get_color()

        plt.fill_between(
            timestamps_rel,
            mean_temp - std_temp,
            mean_temp + std_temp,
            color=color,
            alpha=0.12,
        )

    plt.axhline(
        TARGET_LOW_C,
        linestyle="--",
        linewidth=1.4,
        color="black",
        label="Target low",
    )

    plt.axhline(
        TARGET_HIGH_C,
        linestyle="--",
        linewidth=1.4,
        color="gray",
        label="Target high",
    )

    plt.xlabel("Time (s)")
    plt.ylabel("Temperature (C)")
    plt.title("Active ROI mean temperature with spatial standard deviation")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_all_trials_max_seen_overlay(all_results):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    all_values = []

    for item in all_results:
        all_values.append(item["results"]["max_values"])

    vmin = float(np.nanmin(np.concatenate(all_values)))
    vmax = float(np.nanmax(np.concatenate(all_values)))

    for idx, item in enumerate(all_results):
        results = item["results"]
        points = results["max_points"]
        values = results["max_values"]

        sc = ax.scatter(
            points[:, 0],
            points[:, 1],
            points[:, 2],
            c=values,
            cmap=CMAP,
            s=ROI_POINT_SIZE,
            alpha=0.35,
            vmin=vmin,
            vmax=vmax,
            label=f"Trial {idx + 1}",
        )

    ax.set_title("Max seen temperature across experiment")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Z (m)")

    all_points = np.vstack([item["results"]["max_points"] for item in all_results])
    make_equal_axes_3d(ax, all_points)

    cbar = plt.colorbar(sc, ax=ax, shrink=0.75, pad=0.08)
    cbar.set_label("Temperature (C)")

    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_all_trials_roi_overlay(all_results):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    for idx, item in enumerate(all_results):
        results = item["results"]

        roi_points = np.vstack(results["roi_xyz_all"])

        ax.scatter(
            roi_points[:, 0],
            roi_points[:, 1],
            roi_points[:, 2],
            s=ROI_POINT_SIZE,
            alpha=0.35,
            label=f"Trial {idx + 1}",
        )

    ax.set_title("ROI overlay pointcloud")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Z (m)")

    all_points = np.vstack([
        np.vstack(item["results"]["roi_xyz_all"])
        for item in all_results
    ])

    make_equal_axes_3d(ax, all_points)

    ax.legend()
    plt.tight_layout()
    plt.show()


def run_analysis():
    all_results = []

    for npz_path in NPZ_PATHS:
        print("")
        print("============================================================")
        print("Running analysis")
        print("============================================================")
        print(npz_path)

        if not os.path.exists(npz_path):
            print(f"File not found, skipping: {npz_path}")
            continue

        try:
            results = prepare_dataset(npz_path)
            print_diagnostics(results)

            all_results.append({
                "path": npz_path,
                "results": results,
            })

        except Exception as exc:
            print("Analysis failed:")
            print(exc)

    if len(all_results) == 0:
        print("No valid results to plot.")
        return all_results

    plot_all_trials_roi_overlay(all_results)
    plot_all_trials_max_seen_overlay(all_results)
    plot_all_trials_active_percent_in_band(all_results)
    plot_all_trials_active_mean_temp_with_std(all_results)

    return all_results


all_results = run_analysis()

ModuleNotFoundError: No module named 'matplotlib'